# Lab 19 — MITRE ATLAS & AI Threat Modeling

*AI Cybersecurity Masterclass · Lab 19 · AI60secLab*

**Companion video:** [▶ MITRE ATLAS: The AI-Attack Playbook](https://www.youtube.com/watch?v=QAUlOZvrlQY&list=PLG4l6Ssi9A00)

**Research question:** How does the threat model change when the protected system contains machine-learning or generative-AI components?

This lab downloads official MITRE ATLAS STIX data and converts AI-focused techniques into an analyzable table.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
from src.mitre_data import (
    ATLAS_STIX_URL,
    load_remote_or_local,
    technique_catalog,
)

local_fallback = DATA_DIR / "stix-atlas.json"

atlas_bundle = load_remote_or_local(
    ATLAS_STIX_URL,
    local_path=local_fallback,
)

atlas_techniques = technique_catalog(
    atlas_bundle,
    id_prefix="AML.",
)

print(
    "Active ATLAS techniques/sub-techniques:",
    len(atlas_techniques),
)
display(atlas_techniques.head())

In [ ]:
tactic_rows = atlas_techniques.copy()
tactic_rows["tactic"] = tactic_rows["tactics"].str.split(", ")
tactic_rows = tactic_rows.explode("tactic")
tactic_rows = tactic_rows[
    tactic_rows["tactic"].notna()
    & (tactic_rows["tactic"] != "")
]

tactic_counts = (
    tactic_rows.groupby("tactic", as_index=False)
    .agg(technique_count=("external_id", "nunique"))
    .sort_values("technique_count", ascending=False)
)

display(tactic_counts)

In [ ]:
terms = [
    "prompt",
    "poison",
    "model",
    "inference",
    "training",
    "supply chain",
]

for term in terms:
    mask = (
        atlas_techniques["name"].str.contains(
            term, case=False, na=False
        )
        | atlas_techniques["description"].str.contains(
            term, case=False, na=False
        )
    )

    matches = atlas_techniques.loc[
        mask,
        ["external_id", "name", "tactics"]
    ]

    print(f"\nTERM: {term} | matches: {len(matches)}")
    display(matches.head(10))

## Threat-model exercise

Choose one system:
- security classifier;
- RAG assistant;
- autonomous AI agent;
- model-training pipeline.

Select five relevant ATLAS techniques and document:

**Asset → Technique → Preconditions → Security impact → Observable signal → Mitigation hypothesis**